# Lab 8 — ResNet-50 Transfer Learning on DermaMNIST
Streamlined Colab edition. Full commented version is in Google Drive. Educational use only.

In [ ]:
!pip -q install medmnist
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms,models
import medmnist
from medmnist import INFO
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score
import numpy as np
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
flag='dermamnist'; info=INFO[flag]; DataClass=getattr(medmnist,info['python_class']); n_classes=len(info['label'])
tf=transforms.Compose([transforms.Resize((64,64)),transforms.ToTensor(),transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train=DataClass(split='train',transform=tf,download=True,as_rgb=True); val=DataClass(split='val',transform=tf,download=True,as_rgb=True); test=DataClass(split='test',transform=tf,download=True,as_rgb=True)
tr=DataLoader(train,batch_size=64,shuffle=True); te=DataLoader(test,batch_size=128)

In [ ]:
model=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,n_classes); model=model.to(device); loss_fn=nn.CrossEntropyLoss(); opt=optim.SGD(model.fc.parameters(),lr=0.01,momentum=0.9)

In [ ]:
for epoch in range(2):
 model.train(); total=correct=0
 for x,y in tr:
  x=x.to(device); y=y.squeeze().long().to(device); opt.zero_grad(); z=model(x); loss=loss_fn(z,y); loss.backward(); opt.step(); correct+=(z.argmax(1)==y).sum().item(); total+=len(y)
 print('epoch',epoch+1,'accuracy',correct/total)

In [ ]:
model.eval(); yt=[]; yp=[]; probs=[]
with torch.no_grad():
 for x,y in te:
  z=model(x.to(device)); p=torch.softmax(z,1).cpu().numpy(); yt.extend(y.squeeze().numpy()); yp.extend(z.argmax(1).cpu().numpy()); probs.extend(p)
print('Accuracy',accuracy_score(yt,yp)); print('Macro F1',f1_score(yt,yp,average='macro')); print('Macro ROC-AUC',roc_auc_score(yt,np.asarray(probs),multi_class='ovr',average='macro'))